# Model dự báo MG95: XGBoost, LightGBM, Random Forest, LSTM

Notebook này dùng dữ liệu đã chia sẵn:

```text
Data_Train_Val_Test/
    MG95_t_plus_1/
    MG95_t_plus_3/
    MG95_t_plus_7/
```

Mỗi bộ gồm `train.csv`, `val.csv`, `test.csv`.

Mục tiêu:
- Chạy 4 model: **XGBoost**, **LightGBM**, **Random Forest**, **LSTM**
- Chạy cho cả 3 horizon: **t+1**, **t+3**, **t+7**
- Có đủ metrics trên **train / val / test** để kiểm tra overfitting.


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# Nếu chạy trên Google Colab, mở Drive trước
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Cài thư viện cần thiết
# Colab thường đã có sẵn tensorflow. Nếu thiếu xgboost/lightgbm thì cell này sẽ cài thêm.
!pip -q install xgboost lightgbm


In [17]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


## 1. Cấu hình đường dẫn

Nếu thư mục của bạn khác, chỉ cần sửa `BASE_DIR`.

Theo pipeline trước đó, kết quả chia train/val/test đang nằm trong:

```text
/content/drive/MyDrive/DACS_2/data/Data_Train_Val_Test
```


In [18]:
BASE_DIR = "/content/drive/MyDrive/DACS_2/data/Data_Train_Val_Test"

HORIZONS = [1, 3, 7]

print("BASE_DIR tồn tại chưa:", os.path.exists(BASE_DIR))

if os.path.exists(BASE_DIR):
    print("Các thư mục/file bên trong:")
    for item in os.listdir(BASE_DIR):
        print("-", item)
else:
    raise FileNotFoundError("Không tìm thấy BASE_DIR. Hãy kiểm tra lại đường dẫn.")


BASE_DIR tồn tại chưa: True
Các thư mục/file bên trong:
- MG95_t_plus_1
- MG95_t_plus_3
- MG95_t_plus_7
- model_results_all_horizons.csv


## 2. Hàm tính metrics

In [19]:
def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    mask = y_true != 0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_%": mape,
        "R2": r2
    }


## 3. Hàm đọc dữ liệu train/val/test

In [20]:
def load_split_data(horizon, date_col_name_override=None):
    folder = os.path.join(BASE_DIR, f"MG95_t_plus_{horizon}")

    train_path = os.path.join(folder, "train.csv")
    val_path = os.path.join(folder, "val.csv")
    test_path = os.path.join(folder, "test.csv")

    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Không tìm thấy: {train_path}")
    if not os.path.exists(val_path):
        raise FileNotFoundError(f"Không tìm thấy: {val_path}")
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Không tìm thấy: {test_path}")

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)

    # Determine the actual date column name
    actual_date_col_name = None
    if date_col_name_override:
        # Check if the override column exists in all dataframes
        if all(date_col_name_override in df.columns for df in [train_df, val_df, test_df]):
            actual_date_col_name = date_col_name_override
        else:
            # Provide more specific error for override case
            missing_dfs = [idx for idx, df in enumerate([train_df, val_df, test_df]) if date_col_name_override not in df.columns]
            df_names = ["train_df", "val_df", "test_df"]
            missing_df_names = ", ".join([df_names[i] for i in missing_dfs])
            raise KeyError(f"Cột ngày tháng '{date_col_name_override}' không tìm thấy trong {missing_df_names}. Các cột hiện có trong train_df: {train_df.columns.tolist()}")
    else:
        # Default check for 'Date' or 'date'
        if "Date" in train_df.columns and "Date" in val_df.columns and "Date" in test_df.columns:
            actual_date_col_name = "Date"
        elif "date" in train_df.columns and "date" in val_df.columns and "date" in test_df.columns:
            actual_date_col_name = "date"
        else:
            raise KeyError(f"Không tìm thấy cột 'Date' hay 'date' trong tất cả các dataframe. Hãy kiểm tra lại file CSV hoặc chỉ định tên cột ngày tháng bằng tham số 'date_col_name_override'. Các cột hiện có trong train_df: {train_df.columns.tolist()}")

    for df in [train_df, val_df, test_df]:
        df[actual_date_col_name] = pd.to_datetime(df[actual_date_col_name])
        df.sort_values(actual_date_col_name, inplace=True)
        df.reset_index(drop=True, inplace=True)

    target_col = f"MG95_t_plus_{horizon}"
    feature_cols = [c for c in train_df.columns if c not in [actual_date_col_name, target_col]]

    # Return actual_date_col_name as well
    return train_df, val_df, test_df, feature_cols, target_col, actual_date_col_name

## 4. Model tabular: XGBoost, LightGBM, Random Forest

Ba model này dùng trực tiếp bảng feature đã tạo sau merge.


In [21]:
def get_tabular_models():
    models = {
        "XGBoost": XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "LightGBM": LGBMRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=15,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            verbose=-1
        ),

        "RandomForest": RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }

    return models


## 5. Hàm tạo sequence cho LSTM

LSTM cần dữ liệu dạng 3 chiều:

```text
samples × timesteps × features
```

Ở đây dùng `SEQ_LEN = 10`, tức mỗi mẫu dùng 10 dòng quan sát liên tiếp trước đó.


In [22]:
SEQ_LEN = 10

def make_sequences(X, y, dates, seq_len=SEQ_LEN):
    X_seq, y_seq, date_seq = [], [], []

    X_values = np.asarray(X)
    y_values = np.asarray(y).reshape(-1)
    date_values = np.asarray(dates)

    for i in range(seq_len, len(X_values)):
        X_seq.append(X_values[i-seq_len:i])
        y_seq.append(y_values[i])
        date_seq.append(date_values[i])

    return np.array(X_seq), np.array(y_seq), np.array(date_seq)


def build_lstm_model(n_timesteps, n_features):
    model = Sequential([
        LSTM(32, input_shape=(n_timesteps, n_features)),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="mse"
    )

    return model


## 6. Chạy model cho một horizon

In [25]:
def run_one_horizon(horizon):
    print("=" * 80)
    print(f"ĐANG CHẠY HORIZON t+{horizon}")
    print("=" * 80)

    # Pass date_col_name_override if your date column is named differently
    # Example: train_df, val_df, test_df, feature_cols, target_col, date_col = load_split_data(horizon, date_col_name_override="MyDateColumn")
    # Hãy thay thế "YOUR_DATE_COLUMN_NAME" bằng tên cột ngày tháng thực tế trong CSV của bạn.
    train_df, val_df, test_df, feature_cols, target_col, date_col = load_split_data(horizon, date_col_name_override="YOUR_DATE_COLUMN_NAME")

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()

    X_val = val_df[feature_cols].copy()
    y_val = val_df[target_col].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].copy()

    results = []
    predictions = []

    # =========================
    # 1. XGBoost / LightGBM / RF
    # =========================
    tabular_models = get_tabular_models()

    for model_name, model in tabular_models.items():
        print(f"Training {model_name}...")

        model.fit(X_train, y_train)

        pred_train = model.predict(X_train)
        pred_val = model.predict(X_val)
        pred_test = model.predict(X_test)

        for split_name, y_true, y_pred in [
            ("train", y_train, pred_train),
            ("val", y_val, pred_val),
            ("test", y_test, pred_test)
        ]:
            m = calc_metrics(y_true, y_pred)
            results.append({
                "Horizon": horizon,
                "Model": model_name,
                "Split": split_name,
                **m
            })

        pred_df = pd.DataFrame({
            "Date": test_df[date_col],
            "Horizon": horizon,
            "Model": model_name,
            "Actual": y_test.values,
            "Predicted": pred_test
        })
        predictions.append(pred_df)

    # =========================
    # 2. LSTM
    # =========================
    print("Training LSTM...")

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_test_scaled = scaler_X.transform(X_test)

    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).reshape(-1)
    y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1)).reshape(-1)
    y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).reshape(-1)

    X_train_seq, y_train_seq, train_dates_seq = make_sequences(
        X_train_scaled, y_train_scaled, train_df[date_col], SEQ_LEN
    )
    X_val_seq, y_val_seq, val_dates_seq = make_sequences(
        X_val_scaled, y_val_scaled, val_df[date_col], SEQ_LEN
    )
    X_test_seq, y_test_seq, test_dates_seq = make_sequences(
        X_test_scaled, y_test_scaled, test_df[date_col], SEQ_LEN
    )

    if len(X_train_seq) == 0 or len(X_val_seq) == 0 or len(X_test_seq) == 0:
        print("Không đủ dữ liệu để chạy LSTM với SEQ_LEN =", SEQ_LEN)
    else:
        lstm_model = build_lstm_model(
            n_timesteps=X_train_seq.shape[1],
            n_features=X_train_seq.shape[2]
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )

        history = lstm_model.fit(
            X_train_seq,
            y_train_seq,
            validation_data=(X_val_seq, y_val_seq),
            epochs=100,
            batch_size=32,
            callbacks=[early_stop],
            verbose=0
        )

        pred_train_scaled = lstm_model.predict(X_train_seq, verbose=0).reshape(-1)
        pred_val_scaled = lstm_model.predict(X_val_seq, verbose=0).reshape(-1)
        pred_test_scaled = lstm_model.predict(X_test_seq, verbose=0).reshape(-1)

        pred_train = scaler_y.inverse_transform(pred_train_scaled.reshape(-1, 1)).reshape(-1)
        pred_val = scaler_y.inverse_transform(pred_val_scaled.reshape(-1, 1)).reshape(-1)
        pred_test = scaler_y.inverse_transform(pred_test_scaled.reshape(-1, 1)).reshape(-1)

        y_train_true = scaler_y.inverse_transform(y_train_seq.reshape(-1, 1)).reshape(-1)
        y_val_true = scaler_y.inverse_transform(y_val_seq.reshape(-1, 1)).reshape(-1)
        y_test_true = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).reshape(-1)

        for split_name, y_true, y_pred in [
            ("train", y_train_true, pred_train),
            ("val", y_val_true, pred_val),
            ("test", y_test_true, pred_test)
        ]:
            m = calc_metrics(y_true, y_pred)
            results.append({
                "Horizon": horizon,
                "Model": "LSTM",
                "Split": split_name,
                **m
            })

        pred_df = pd.DataFrame({
            "Date": pd.to_datetime(test_dates_seq),
            "Horizon": horizon,
            "Model": "LSTM",
            "Actual": y_test_true,
            "Predicted": pred_test
        })
        predictions.append(pred_df)

    results_df = pd.DataFrame(results)
    predictions_df = pd.concat(predictions, ignore_index=True)

    return results_df, predictions_df

## 7. Chạy cả 3 horizon: t+1, t+3, t+7

In [26]:
all_results = []
all_predictions = []

for horizon in HORIZONS:
    result_h, pred_h = run_one_horizon(horizon)
    all_results.append(result_h)
    all_predictions.append(pred_h)

results_all = pd.concat(all_results, ignore_index=True)
predictions_all = pd.concat(all_predictions, ignore_index=True)

display(results_all)


ĐANG CHẠY HORIZON t+1


KeyError: "Cột ngày tháng 'YOUR_DATE_COLUMN_NAME' không tìm thấy trong train_df, val_df, test_df. Các cột hiện có trong train_df: ['MG95_lag1', 'MG95_lag3', 'MG95_lag7', 'MG95_return1', 'MG95_return5', 'MG95_MA5_lag1', 'MG95_STD20_lag1', 'BRT_KH_return1', 'WTI_return1', 'GPRD_lag1', 'GPRD_lag3', 'GPRD_lag7', 'GPRD_MA7_lag1', 'Has_GPR_Event', 'NgayLe', 'GanNgayLe', 'SuKienDacBiet', 'IsMonthStart', 'IsMonthEnd', 'Month', 'Quarter', 'DayOfWeek', 'MG95_t_+_1']"

## 8. Bảng kết quả dạng dễ đọc

In [ ]:
summary_table = results_all.pivot_table(
    index=["Horizon", "Model"],
    columns="Split",
    values=["MAE", "RMSE", "MAPE_%", "R2"]
)

display(summary_table)

output_results = os.path.join(BASE_DIR, "model_results_XGB_LGBM_RF_LSTM_all_horizons.csv")
output_predictions = os.path.join(BASE_DIR, "test_predictions_XGB_LGBM_RF_LSTM_all_horizons.csv")

results_all.to_csv(output_results, index=False, encoding="utf-8-sig")
predictions_all.to_csv(output_predictions, index=False, encoding="utf-8-sig")

print("Đã lưu kết quả:", output_results)
print("Đã lưu dự báo test:", output_predictions)


## 9. Kiểm tra overfitting

Cách đọc nhanh:

```text
Train RMSE rất thấp, nhưng Val/Test RMSE cao nhiều → overfitting.
Train/Val/Test gần nhau → model ổn hơn.
Test tốt nhưng Val kém → có thể giai đoạn test dễ hơn hoặc dữ liệu biến động khác.
```


In [ ]:
overfit_check = results_all.pivot_table(
    index=["Horizon", "Model"],
    columns="Split",
    values="RMSE"
).reset_index()

overfit_check["Val/Train_RMSE"] = overfit_check["val"] / overfit_check["train"]
overfit_check["Test/Train_RMSE"] = overfit_check["test"] / overfit_check["train"]

display(overfit_check.sort_values(["Horizon", "Test/Train_RMSE"]))


## 10. Chọn model tốt nhất theo Validation RMSE

In [ ]:
val_results = results_all[results_all["Split"] == "val"].copy()

best_models = (
    val_results
    .sort_values(["Horizon", "RMSE"])
    .groupby("Horizon")
    .head(1)
    .reset_index(drop=True)
)

display(best_models)


## 11. Vẽ Actual vs Predicted trên test cho model tốt nhất mỗi horizon

In [ ]:
for _, row in best_models.iterrows():
    horizon = row["Horizon"]
    model_name = row["Model"]

    plot_df = predictions_all[
        (predictions_all["Horizon"] == horizon) &
        (predictions_all["Model"] == model_name)
    ].copy()

    plot_df = plot_df.sort_values("Date")

    plt.figure(figsize=(12, 5))
    plt.plot(plot_df["Date"], plot_df["Actual"], label="Actual")
    plt.plot(plot_df["Date"], plot_df["Predicted"], label="Predicted")
    plt.title(f"MG95 t+{horizon} - {model_name} - Test set")
    plt.xlabel("Date")
    plt.ylabel("MG95")
    plt.legend()
    plt.grid(True)
    plt.show()
